In [ ]:
# Run this cell first, both locally and on Kaggle.
from pathlib import Path
import importlib
import subprocess
import sys

GITHUB_REPO = "https://github.com/ngvihoa/boundary-unlearning.git"  
GITHUB_REF = "main"  # Branch, tag, or commit SHA. Use a SHA to repeat an experiment.
REPO_DIR = Path("/kaggle/working/boundary-unlearning")

if GITHUB_REPO:
    if not (REPO_DIR / ".git").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--no-checkout",
            GITHUB_REPO, str(REPO_DIR),
        ], check=True)
    subprocess.run([
        "git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", GITHUB_REF,
    ], check=True)
    subprocess.run([
        "git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD",
    ], check=True)
    CODE_ROOT = REPO_DIR
else:
    CODE_ROOT = Path.cwd()

if not (CODE_ROOT / "boundary_unlearning" / "__init__.py").is_file():
    raise FileNotFoundError(
        "Set GITHUB_REPO to your GitHub HTTPS URL on Kaggle, "
        "or run this notebook from the local repository root."
    )

sys.path.insert(0, str(CODE_ROOT))
importlib.invalidate_caches()
print("Code root:", CODE_ROOT)
if (CODE_ROOT / ".git").exists():
    print("Code commit:", subprocess.check_output(
        ["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"], text=True,
    ).strip())
# Restart the kernel and Run All after changing GITHUB_REF to reload the modules.


In [ ]:
import csv
import hashlib
import json
import platform
import random
from pathlib import Path

import numpy as np
import torch
import torchvision
from torch.utils.data import Subset
from torchvision import datasets, transforms
from functools import partial
from IPython.display import display

from boundary_unlearning import data, models, training, unlearning
from boundary_unlearning.constants import CIFAR_CLASSES
from boundary_unlearning.reporting import results_table

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================
RUN_PROFILE = "paper"  # "smoke", "paper", "repo_cli"

METHODS_TO_RUN = [
    "boundary_shrink",
    "boundary_expanding"
]

FORGET_CLASS = 4
SEED = 2022
BATCH_SIZE = 64
NUM_WORKERS = 2

REUSE_CHECKPOINTS = False
DOWNLOAD_CIFAR10 = True  # Official labeled test set; cached under /kaggle/working.
ATTACK_NORMALIZATION = "repo_exact"  # Preserve the author's attack normalization.

FGSM_BOUND = 0.1
UNLEARN_LR = 1e-5
MOMENTUM = 0.9
NUM_CLASSES = 10
# Optional Enhanced Boundary Shrink, matching the author's defaults.
EXTRA_EXP = None  # None, "curv", or "weight_assign"
CURVATURE_LAMBDA = 0.7
REWEIGHT_BIAS = -0.5
REWEIGHT_SLOPE = 5.0
# ============================================================
# 2. TRAINING CONFIGURATION
# ============================================================
if RUN_PROFILE == "smoke":
    ORIGINAL_EPOCHS = 1
    ORIGINAL_LR = 0.1
    UNLEARN_EPOCHS = 1
    MAX_BATCHES_PER_EPOCH = 3
    EVALUATE_TRAIN_SPLITS = False

elif RUN_PROFILE == "paper":
    ORIGINAL_EPOCHS = 30
    ORIGINAL_LR = 0.1
    UNLEARN_EPOCHS = 10
    MAX_BATCHES_PER_EPOCH = None
    EVALUATE_TRAIN_SPLITS = True

elif RUN_PROFILE == "repo_cli":
    ORIGINAL_EPOCHS = 50
    ORIGINAL_LR = 0.01
    UNLEARN_EPOCHS = 10
    MAX_BATCHES_PER_EPOCH = None
    EVALUATE_TRAIN_SPLITS = True

else:
    raise ValueError(
        "RUN_PROFILE must be smoke, paper, or repo_cli"
    )

if ATTACK_NORMALIZATION not in {"matched", "repo_exact"}:
    raise ValueError("Unsupported ATTACK_NORMALIZATION")

if EXTRA_EXP not in {None, "curv", "weight_assign"}:
    raise ValueError("Unsupported EXTRA_EXP")

if not 0 <= FORGET_CLASS < NUM_CLASSES:
    raise ValueError("FORGET_CLASS must be between 0 and 9")

if any(
    method not in {"boundary_shrink", "boundary_expanding"}
    for method in METHODS_TO_RUN
):
    raise ValueError("Unsupported unlearning method")

# ============================================================
# 3. KAGGLE PATHS
# ============================================================
INPUT_ROOT = Path("/kaggle/input")
DATA_ROOT = Path(
    "/kaggle/input/datasets/petitbonney/cifar10-image-recognition"
)

OUTPUT_ROOT = Path("/kaggle/working")
CIFAR10_ROOT = OUTPUT_ROOT / "cifar10"
RESULT_ROOT = OUTPUT_ROOT / "result"
CHECKPOINT_ROOT = RESULT_ROOT / "checkpoints"

# Kaggle Input is read-only; create directories only in working.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# 4. NORMALIZATION
# ============================================================
CIFAR_MEAN = (0.5, 0.5, 0.5)
CIFAR_STD = (0.5, 0.5, 0.5)

REPO_ATTACK_MEAN = (0.4914, 0.4822, 0.2265)
REPO_ATTACK_STD = (0.2023, 0.1994, 0.2010)

# ============================================================
# 5. REPRODUCIBILITY
# ============================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ============================================================
# 6. ENVIRONMENT CHECK
# ============================================================
print("=" * 60)
print("ENVIRONMENT INFORMATION")
print("=" * 60)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("NumPy:", np.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is unavailable. Enable GPU in Kaggle settings.")

print("\nInput directories:")
for path in INPUT_ROOT.iterdir():
    print(" -", path)

print("\nData root exists:", DATA_ROOT.exists())
print("Output root:", OUTPUT_ROOT)
print("Checkpoint root:", CHECKPOINT_ROOT)
print("Result root:", RESULT_ROOT)
print("\nConfiguration initialized successfully.")


In [ ]:
train_images, train_raw = data.load_concatenated_npy_images(DATA_ROOT / "train.npy")
test_images, test_raw = data.load_concatenated_npy_images(DATA_ROOT / "test.npy")
print("Train images:", train_images.shape, train_images.dtype)
print("Test images:", test_images.shape, test_images.dtype)


In [ ]:
train_label_rows, train_label_columns = data.read_csv_rows(DATA_ROOT / "trainLabels.csv")
train_label_col = data.required_column(train_label_columns, "label", DATA_ROOT / "trainLabels.csv")
train_id_col = data.required_column(train_label_columns, "id", DATA_ROOT / "trainLabels.csv")
train_labels = data.encode_cifar_labels([row[train_label_col] for row in train_label_rows], num_classes=NUM_CLASSES)
train_ids = [data.parse_numeric_id(row[train_id_col]) for row in train_label_rows]
if all(value is None for value in train_ids):
    print("Warning: No numeric IDs found in training labels.")

sample_rows, sample_columns = data.read_csv_rows(DATA_ROOT / "sampleSubmission.csv")
sample_id_col = data.required_column(sample_columns, "id", DATA_ROOT / "sampleSubmission.csv")
sample_label_col = data.required_column(sample_columns, "label", DATA_ROOT / "sampleSubmission.csv")
sample_ids = [data.parse_numeric_id(row[sample_id_col]) for row in sample_rows]
sample_labels = [row[sample_label_col] for row in sample_rows]

print("Train label columns:", list(train_label_columns.keys()))
print("Number of training labels:", len(train_labels))
print("Number of training IDs:", len(train_ids))
print("Training sample rows:", train_label_rows[:5])
print("Sample submission columns:", list(sample_columns.keys()))
print("Number of sample submission rows:", len(sample_rows))
print("Sample submission rows:", sample_rows[:5])
print("Encoded training labels:", train_labels[:5])
print("Parsed training IDs:", train_ids[:5])


In [ ]:
data.validate_image_array(train_images, "train.npy")
data.validate_image_array(test_images, "test.npy")

if len(train_images) != len(train_labels):
    raise ValueError(f"Training image/label count mismatch: {len(train_images)} images vs {len(train_labels)} labels.")
if len(test_images) != len(sample_ids):
    raise ValueError(f"Test image/sample submission count mismatch: {len(test_images)} images vs {len(sample_ids)} sample IDs.")

print("Image arrays validated successfully.")
print("Train images:", train_images.shape)
print("Train labels:", train_labels.shape)
print("Test images:", test_images.shape)
print("Sample submission IDs:", len(sample_ids))


In [ ]:
train_dataset = data.CIFAR10NpyDataset(train_images, train_labels)
test_inference_dataset = data.CIFAR10NpyDataset(test_images)

# The Kaggle competition test arrays have no ground-truth labels.
# Match the author by evaluating on the separate official CIFAR-10 test set.
evaluation_dataset = datasets.CIFAR10(
    root=str(CIFAR10_ROOT),
    train=False,
    download=DOWNLOAD_CIFAR10,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]),
)

train_fit_set = train_dataset
train_forget_indices = np.flatnonzero(train_labels == FORGET_CLASS).tolist()
train_retain_indices = np.flatnonzero(train_labels != FORGET_CLASS).tolist()
test_labels = np.asarray(evaluation_dataset.targets, dtype=np.int64)
test_forget_indices = np.flatnonzero(test_labels == FORGET_CLASS).tolist()
test_retain_indices = np.flatnonzero(test_labels != FORGET_CLASS).tolist()

train_forget_set = Subset(train_dataset, train_forget_indices)
train_retain_set = Subset(train_dataset, train_retain_indices)
test_forget_set = Subset(evaluation_dataset, test_forget_indices)
test_retain_set = Subset(evaluation_dataset, test_retain_indices)
test_inference_set = test_inference_dataset

print("Train fit:", len(train_fit_set))
print("Train forget:", len(train_forget_set))
print("Train retain:", len(train_retain_set))
print("Official test:", len(evaluation_dataset))
print("Official test forget:", len(test_forget_set))
print("Official test retain:", len(test_retain_set))
print("Kaggle test inference:", len(test_inference_set))


In [ ]:
PIN_MEMORY = DEVICE.type == "cuda"
make_loader = partial(
    data.make_loader, batch_size=BATCH_SIZE, seed=SEED,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)

train_all_loader = make_loader(train_fit_set, shuffle=True)
train_retain_loader = make_loader(train_retain_set, shuffle=True)
train_retain_eval_loader = make_loader(train_retain_set, shuffle=False)
train_forget_eval_loader = make_loader(train_forget_set, shuffle=False)

test_loader = make_loader(evaluation_dataset, shuffle=False)
test_forget_loader = make_loader(test_forget_set, shuffle=False)
test_retain_loader = make_loader(test_retain_set, shuffle=False)
test_inference_loader = make_loader(test_inference_set, shuffle=False)

class_count = np.bincount(train_labels, minlength=NUM_CLASSES)

print("Class counts:", dict(zip(CIFAR_CLASSES, class_count.tolist())))
print("Train all batches:", len(train_all_loader))
print("Train retain batches:", len(train_retain_loader))
print("Train retain evaluation batches:", len(train_retain_eval_loader))
print("Train forget evaluation batches:", len(train_forget_eval_loader))
print("Official test batches:", len(test_loader))
print("Official test forget batches:", len(test_forget_loader))
print("Official test retain batches:", len(test_retain_loader))
print("Kaggle test inference batches:", len(test_inference_loader))


In [ ]:
# ============================================================
# PHASE 3: MODEL ARCHITECTURE
# ============================================================

shape_model = models.AllCNN().to(DEVICE)
shape_model.apply(models.init_linear_like_repo)
shape_model.eval()

with torch.no_grad():
    sample_images = torch.randn(2, 3, 32, 32, device=DEVICE)
    sample_logits = shape_model(sample_images)

print("Device:", DEVICE)
print("Model:", shape_model.__class__.__name__)
print("Sample input shape:", sample_images.shape)
print("Sample logits shape:", sample_logits.shape)
print("Total parameters:", sum(p.numel() for p in shape_model.parameters()))
assert sample_logits.shape == (2, NUM_CLASSES)

del shape_model, sample_images, sample_logits


In [ ]:
create_or_load_model = partial(
    training.create_or_load_model, device=DEVICE,
    reuse_checkpoints=REUSE_CHECKPOINTS, num_classes=NUM_CLASSES,
    epochs=ORIGINAL_EPOCHS, learning_rate=ORIGINAL_LR, momentum=MOMENTUM,
    max_batches_per_epoch=MAX_BATCHES_PER_EPOCH,
)


In [ ]:
tag_settings = {
    "profile": RUN_PROFILE,
    "data_root": str(DATA_ROOT),
    "implementation_version": "author_alignment_v1",
    "evaluation_dataset": "official_cifar10_test",
    "checkpoint_selection": "best_test_accuracy",
    "extra_exp": EXTRA_EXP,
    "curvature_lambda": CURVATURE_LAMBDA,
    "reweight_bias": REWEIGHT_BIAS,
    "reweight_slope": REWEIGHT_SLOPE,
    "method": METHODS_TO_RUN,
    "forget_class": FORGET_CLASS,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "original_epochs": ORIGINAL_EPOCHS,
    "original_lr": ORIGINAL_LR,
    "unlearn_epochs": UNLEARN_EPOCHS,
    "unlearn_lr": UNLEARN_LR,
    "momentum": MOMENTUM,
    "fgsm_bound": FGSM_BOUND,
    "attack_normalization": ATTACK_NORMALIZATION,
    "max_batches_per_epoch": MAX_BATCHES_PER_EPOCH,
}

tag_hash = hashlib.sha256(
    json.dumps(tag_settings, sort_keys=True).encode("utf-8")
).hexdigest()[:8]

run_tag = f"{RUN_PROFILE}_forget{FORGET_CLASS}_{tag_hash}"

original_checkpoint = CHECKPOINT_ROOT / f"original_{run_tag}.pt"
retrain_checkpoint = CHECKPOINT_ROOT / f"retrain_{run_tag}.pt"

print("Run tag:", run_tag)
print("Original checkpoint:", original_checkpoint)
print("Retrain checkpoint:", retrain_checkpoint)

original_model, original_train_seconds = create_or_load_model(
    "original",
    train_all_loader,
    original_checkpoint,
    test_loader,
)


In [ ]:
retrain_model, retrain_train_seconds = create_or_load_model(
    "retrain",
    train_retain_loader,
    retrain_checkpoint,
    test_retain_loader,
)

print("Original model ready:", original_model is not None)
print("Retrain model ready:", retrain_model is not None)
print("Original training time:", original_train_seconds)
print("Retrain training time:", retrain_train_seconds)


In [ ]:
run_boundary_shrink = partial(
    unlearning.run_boundary_shrink, device=DEVICE, epochs=UNLEARN_EPOCHS,
    learning_rate=UNLEARN_LR, momentum=MOMENTUM, pin_memory=PIN_MEMORY,
    max_batches_per_epoch=MAX_BATCHES_PER_EPOCH,
    bound=FGSM_BOUND, normalization_mode=ATTACK_NORMALIZATION,
    extra_exp=EXTRA_EXP, curvature_lambda=CURVATURE_LAMBDA,
    reweight_bias=REWEIGHT_BIAS, reweight_slope=REWEIGHT_SLOPE,
    cifar_mean=CIFAR_MEAN, cifar_std=CIFAR_STD,
    attack_mean=REPO_ATTACK_MEAN, attack_std=REPO_ATTACK_STD,
)


In [ ]:
run_boundary_expanding = partial(
    unlearning.run_boundary_expanding, device=DEVICE, epochs=UNLEARN_EPOCHS,
    learning_rate=UNLEARN_LR, momentum=MOMENTUM, num_classes=NUM_CLASSES,
    pin_memory=PIN_MEMORY, max_batches_per_epoch=MAX_BATCHES_PER_EPOCH,
)


In [ ]:
#PHASE 6: RUN UNLEARNING METHOD
unlearned_models = {}
unlearn_stats = {}

for method_name in METHODS_TO_RUN:
    method_checkpoint = CHECKPOINT_ROOT / f"{method_name}_{run_tag}.pt"
    method_forget_loader = make_loader(train_forget_set, shuffle = True)

    if method_name == "boundary_shrink":
        model, stats = run_boundary_shrink(original_model, method_forget_loader, method_checkpoint)
    elif method_name == "boundary_expanding":
        model, stats = run_boundary_expanding(original_model, method_forget_loader, method_checkpoint)
    else:
        raise ValueError(f"Unsupported unlearning method: {method_name}")

    unlearned_models[method_name] = model
    unlearn_stats[method_name] = stats

print("Finish methods:", list(unlearned_models))


In [ ]:
evaluate_model_splits = partial(
    training.evaluate_model_splits, device=DEVICE,
    test_loader=test_loader, test_retain_loader=test_retain_loader,
    test_forget_loader=test_forget_loader,
    train_retain_eval_loader=train_retain_eval_loader,
    train_forget_eval_loader=train_forget_eval_loader,
)

# ============================================================
# PHASE 7: OFFICIAL CIFAR-10 TEST EVALUATION
# ============================================================

models_to_evaluate = {
    "original": original_model,
    "retrain": retrain_model,
    **unlearned_models,
}

rows = []
baseline_train_times = {"original": original_train_seconds, "retrain": retrain_train_seconds}

for name, model in models_to_evaluate.items():
    row = {"model": name}
    row.update(evaluate_model_splits(model, include_train=EVALUATE_TRAIN_SPLITS))

    if name in baseline_train_times:
        row["training_seconds"] = baseline_train_times[name]
    else:
        row["training_seconds"] = None

    row["unlearning_seconds"] = unlearn_stats[name]["seconds"] if name in unlearn_stats else None
    row["fgsm_cross_rate"] = unlearn_stats[name].get("cross_rate") if name in unlearn_stats else None
    row["unlearning_update_steps"] = unlearn_stats[name].get("update_steps") if name in unlearn_stats else None

    rows.append(row)

display(results_table(rows))

metrics_path = RESULT_ROOT / f"{run_tag}_metrics.csv"
with metrics_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

config = {
    "run_tag_settings": tag_settings,
    "run_profile": RUN_PROFILE,
    "data_root": str(DATA_ROOT),
    "methods": METHODS_TO_RUN,
    "forget_class": FORGET_CLASS,
    "forget_class_name": CIFAR_CLASSES[FORGET_CLASS],
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "evaluation_dataset": "official_cifar10_test",
    "train_sample_count": len(train_fit_set),
    "test_sample_count": len(evaluation_dataset),
    "extra_exp": EXTRA_EXP,
    "curvature_lambda": CURVATURE_LAMBDA,
    "reweight_bias": REWEIGHT_BIAS,
    "reweight_slope": REWEIGHT_SLOPE,
    "original_epochs": ORIGINAL_EPOCHS,
    "original_learning_rate": ORIGINAL_LR,
    "unlearn_epochs": UNLEARN_EPOCHS,
    "unlearn_learning_rate": UNLEARN_LR,
    "fgsm_bound": FGSM_BOUND,
    "attack_normalization": ATTACK_NORMALIZATION,
    "train_npy_shape": list(train_images.shape),
    "test_npy_shape": list(test_images.shape),
    "sample_submission_rows": len(sample_rows),
    "device": str(DEVICE),
}

config_path = RESULT_ROOT / f"{run_tag}_config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

print("\nMetrics saved to:", metrics_path)
print("Configuration saved to:", config_path)
print("Checkpoint folder:", CHECKPOINT_ROOT)


In [ ]:
## ONLY USE FOR UPDATE CODE FROM GIT WITHOUT RESTART KERNEL
# !git -C /kaggle/working/boundary-unlearning fetch --depth 1 origin main
# !git -C /kaggle/working/boundary-unlearning checkout --detach FETCH_HEAD

In [ ]:
# ============================================================
# EXPORT RESULTS: run this cell independently to refresh the ZIP.
# ============================================================
from pathlib import Path
import os
from IPython.display import FileLink, display
from boundary_unlearning.artifacts import export_results
from boundary_unlearning.reporting import results_table

# Use the singular folder name even in a session using the older path settings.
result_dir = Path(OUTPUT_ROOT) / "result"
result_dir.mkdir(parents=True, exist_ok=True)
table_path = result_dir / f"{run_tag}_results.html"
results_table(rows).to_html(table_path)

archive_path = export_results(
    result_dir,
    checkpoint_dir=CHECKPOINT_ROOT,
    files=[metrics_path, config_path],
)
print(f"Archive: {archive_path} ({archive_path.stat().st_size / (1024 ** 2):.2f} MiB)")
display(FileLink(os.path.relpath(archive_path), result_html_prefix="Download: "))
